# Add external scoring to agentic search loop

This notebook has a basic agentic search loop

* We have a set of furniture in our catalog
* We tell the Agent our preferences
* The agent uses the search tool to recommend furniture

In this notebook we mostly get a feel for how the overall loop works by unrolling it step by step

## New in this notebook

* We add access to external relevance info with a scoring tool
* Here we _cheat_ by using the labeled data from this dataset
* You would put your LTR or other scoring model here

## Things to to notice

* We talk to the LLM in terms of user satisfaction, not relevance. LLMs want to please users!

In [ ]:
import os
os.environ["CHEAT_AT_SEARCH_DATA_PATH"] = "/home/jovyan/data"

from cheat_at_search.data_dir import mount
mount(use_gdrive=False)    # colab, share data across notebook runs on gdrive
# mount(use_gdrive=False) # <- colab without gdrive
# mount(use_gdrive=False, manual_path="/path/to/directory")  # <- force data path to specific directory, ie you're running locally.

  Cloning https://github.com/softwaredoug/cheat-at-search.git to /tmp/pip-req-build-zg5147h3
  Running command git clone --filter=blob:none --quiet https://github.com/softwaredoug/cheat-at-search.git /tmp/pip-req-build-zg5147h3
  Resolved https://github.com/softwaredoug/cheat-at-search.git to commit 04b243764b805c645bd21928e5f142934f5d1f39
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Get an OpenAI Key

This will prompt you for an OpenAI Key to interact with GPT-5

In [ ]:
from cheat_at_search.data_dir import key_for_provider
from openai import OpenAI

OPENAI_KEY = key_for_provider("openai")

openai = OpenAI(api_key=OPENAI_KEY)

## Load the Wayfair corpus

We'll recommend products only from this corpus

In [ ]:
from cheat_at_search.wands_data import corpus
corpus

,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,doc_id,title,description,category,sub_category,cat_subcat,title_snowball,description_snowball
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboa...,"good , deep sleep can be quite difficult to ha...",overallwidth-sidetoside:64.7|dsprimaryproducts...,15.0,4.5,15.0,"[overallwidth-sidetoside:64.7, dsprimaryproduc...",0,solid wood platform bed,"good , deep sleep can be quite difficult to ha...",Furniture,Bedroom Furniture,Furniture / Bedroom Furniture,"Terms({'solid', 'wood', 'platform', 'bed'})","Terms({'there', 'problem', 'space', 'memori', ..."
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,"create delicious slow-cooked meals , from tend...",capacityquarts:7|producttype : slow cooker|pro...,100.0,2.0,98.0,"[capacityquarts:7, producttype : slow cooker, ...",1,all-clad 7 qt . slow cooker,"create delicious slow-cooked meals , from tend...",Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances,"Terms({'all', 'cooker', 'clad', 'slow', '7', '...","Terms({'prepar', 's', 'the', 'walk', 'not', 'u..."
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,prepare home-cooked meals on any schedule with...,features : keep warm setting|capacityquarts:6....,208.0,3.0,181.0,"[features : keep warm setting, capacityquarts:...",2,all-clad electrics 6.5 qt . slow cooker,prepare home-cooked meals on any schedule with...,Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances,"Terms({'all', 'cooker', '6', 'clad', 'slow', '...","Terms({'prepar', 'and', 'ani', 'insert', 'esse..."
3,3,all-clad all professional tools pizza cutter,"Slicers, Peelers And Graters",Browse By Brand / All-Clad,this original stainless tool was designed to c...,overallwidth-sidetoside:3.5|warrantylength : l...,69.0,4.5,42.0,"[overallwidth-sidetoside:3.5, warrantylength :...",3,all-clad all professional tools pizza cutter,this original stainless tool was designed to c...,Browse By Brand,All-Clad,Browse By Brand / All-Clad,"Terms({'pizza', 'all', 'profession', 'cutter',...","Terms({'and', 'origin', 'complement', 's', 'to..."
4,4,baldwin prestige alcott passage knob with roun...,Door Knobs,Home Improvement / Doors & Door Hardware / Doo...,the hardware has a rich heritage of delivering...,compatibledoorthickness:1.375 '' |countryofori...,70.0,5.0,42.0,"[compatibledoorthickness:1.375 '' , countryofo...",4,baldwin prestige alcott passage knob with roun...,the hardware has a rich heritage of delivering...,Home Improvement,Doors & Door Hardware,Home Improvement / Doors & Door Hardware,"Terms({'prestig', 'round', 'rosett', 'baldwin'...","Terms({'entri', 'and', 'prestig', 'ani', 'door..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42989,42989,malibu pressure balanced diverter fixed shower...,Shower Panels,Home Improvement / Bathroom Remodel & Bathroom...,the malibu pressure balanced diverter fixed sh...,producttype : shower panel|spraypattern : rain...,3.0,4.5,2.0,"[producttype : shower panel, spraypattern : ra...",42989,malibu pressure balanced diverter fixed shower...,the malibu pressure balanced diverter fixed sh...,Home Improvement,Bathroom Remodel & Bathroom Fixtures,Home Improvement / Bathroom Remodel & Bathro...,"Terms({'malibu', 'panel', 'pressur', 'fix', 'b...","Terms({'steel', 'and', 'fix', 'ani', 'is', 'th..."
42990,42990,emmeline 5 piece breakfast dining set,Dining Table Sets,Furniture / Kitchen & Dining Furniture / Dinin...,,basematerialdetails : steel| : gray wood|ofhar...,1314.0,4.5,864.0,"[basematerialdetails : steel, : gray wood, of...",42990,emmeline 5 piece breakfast dining set,,Furniture,Kitchen & Dining Furniture,Furniture / Kitchen & Dining Furniture,"Terms({'breakfast', '5', 'set', 'emmelin', 'p

### Index the furniture

We'll index title and description with basic stemming to be able to retrieve them

In [ ]:
from searcharray import SearchArray
from cheat_at_search.tokenizers import snowball_tokenizer

corpus['title_snowball'] = SearchArray.index(corpus['title'].fillna(''), snowball_tokenizer)
corpus['description_snowball'] = SearchArray.index(corpus['description'].fillna(''), snowball_tokenizer)

2026-01-09 15:41:06,762 - searcharray.indexing - INFO - Indexing begins w/ 4 workers


INFO:searcharray.indexing:Indexing begins w/ 4 workers


2026-01-09 15:41:06,783 - searcharray.indexing - INFO - 0 Batch Start tokenization


INFO:searcharray.indexing:0 Batch Start tokenization


2026-01-09 15:41:06,798 - searcharray.indexing - INFO - Tokenizing 42994 documents


...

INFO:searcharray.indexing:Batch tokenization complete


2026-01-09 15:41:28,783 - searcharray.indexing - INFO - (main thread) Processing 1 batch results


INFO:searcharray.indexing:(main thread) Processing 1 batch results


2026-01-09 15:41:28,969 - searcharray.indexing - INFO - Indexing from tokenization complete


INFO:searcharray.indexing:Indexing from tokenization complete


## Create a furniture products search function

Here is a function that searches a Wayfair product dataset. It's just a Python function that returns top 10 pieces of furniture.

Right now we'll call it directly, soon we'll help ChatGPT interact with this.

In [ ]:
import numpy as np
from typing import Union

def search_furniture(keywords: str) -> list[dict[str, Union[str, int, float]]]:
    """Search the available furniture products, get top 10 furniture.

    This is just a naive BM25 / keyword search of the product title and description.
    Don't expect sophisticated synonyms or semantic search. Just basic keyword with
    some stemming.

    """
    print("search", keywords)
    required_keywords = [term[1:] for term in keywords.split() if term.startswith("+")]
    bm25_scores = np.zeros(len(corpus))
    for term in snowball_tokenizer(keywords):
        bm25_scores += corpus['title_snowball'].array.score(term) * 9.3
        bm25_scores += corpus['description_snowball'].array.score(term) * 4.1

    for required_term in snowball_tokenizer(" ".join(required_keywords)):
        required_score = (corpus['title_snowball'].array.score(required_term) +
                          corpus['description_snowball'].array.score(required_term))
        bm25_scores[required_score == 0] = 0

    top_k_indices = np.argsort(bm25_scores)[-10:][::-1]
    bm25_scores = bm25_scores[top_k_indices]
    top_movies = corpus.iloc[top_k_indices].copy()
    top_movies.loc[:, 'score'] = bm25_scores

    results = []
    for id, row in top_movies.iterrows():
        results.append({
            'id': row['doc_id'],
            'title': row['title'],
            'description': row['description'],
            'score': row['score']
        })
    return results



results = search_furniture("gray leather cocktail table")

search gray leather cocktail table


## Add a tool to evaluate

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, Literal
from cheat_at_search.wands_data import judgments

class SearchResult(BaseModel):
    """A furniture search result."""
    doc_id: int = Field(..., description="The product id")
    title: str = Field(..., description="The title of the product (from the corpus)")
    description: str = Field(..., description="The description of (from the corpus)")
    explanation: str = Field(..., description="Why this is relevant for the search in your words")

class SearchResults(BaseModel):
    """Ranked search results in order of relevance."""
    search_results: list[SearchResult] = Field(..., description="The search results")

class ScoredSearchResult(BaseModel):
    """A search result with a score."""
    result: SearchResult = Field(..., description="The search result")
    user_satisfaction: Optional[Literal['😁', '🫤', '😥']] = Field(..., description="How happy this result will make the user")

class ScoredSearchResults(BaseModel):
    """Ranked search results in order of relevance."""
    search_results: list[ScoredSearchResult] = Field(..., description="The search results")
    message: Optional[str] = Field(..., description="A message to the user about the results")
    tool_reflection: Optional[str] = Field(...,
                                           description="What would you tell the next iteration about using the search tool (not specific to this query, just in general)? Format in a way that could be inserted into a prompt")
    tool_queries_satisfying_user: list[str] = Field(...,description="The list of queries you issued to the underlying search tool that yield satisfing results for the user")

def score(query: str, results: SearchResults) -> ScoredSearchResults:
    """Return whether the results for the query will satisfy the user

    * If you call 'score' on a result, and it doesn't have a user_satisfaction label, its unknown whether the user would like it
    * If you get a '😥' then the user doesn't like it
    * If you get a '😁' then the user likes it
    * If you get a '🫤' then the user is meh
    """
    print("score", query, len(results.search_results))
    scored_results = []
    available_queries = judgments['query'].unique()
    skip_score = False
    if query not in available_queries:
        skip_score = True
    scores = judgments[judgments['query'] == query]
    max_score = 2
    for result in results.search_results:
        score = 0
        label = '😥'
        available_doc_ids = scores['doc_id'].unique()
        if result.doc_id in available_doc_ids:
            score = scores[scores['doc_id'] == result.doc_id]['grade'].values[0]
            if score == 2:
                label = '😁'
            elif score == 1:
                label = '🫤'
            elif score == 0:
                label = '😥'
        scored_results.append(ScoredSearchResult(
            result=result,
            user_satisfaction=label if not skip_score else None
        ))
    return ScoredSearchResults(search_results=scored_results,
                               message="",
                               tool_queries_satisfying_user=[],
                               tool_reflection="")

In [ ]:
search_results = []

for result in results:
    search_results.append(SearchResult(
        doc_id=result['id'],
        title=result['title'],
        description=result['description'],
        explanation=str(result['score'])
    ))

search_results = SearchResults(search_results=search_results)
score("gray leather cocktail table", search_results)

score gray leather cocktail table 10


ScoredSearchResults(search_results=[ScoredSearchResult(result=SearchResult(doc_id=24399, title='radersburg leather cocktail table', description="whether you 're serving up trays of tasty crudites at your next elegant gathering or simply looking for a stylish finishing touch to your living room seating ensemble , this chic cocktail table gets the job done . a medium cherry finish pairs with leather-inspired inserts for a touch of rustic charm , while a steel frame and lower storage shelf adds a dash of sleek style and functionality to your space .", explanation='84.16834878921509'), user_satisfaction='🫤'), ScoredSearchResult(result=SearchResult(doc_id=10350, title='boston cocktail table', description='product highlights a round weathered grey casual cocktail table with plank-style details , double “ x ” legs , and an “ x ” base . product information collection : bridgeport dimension : height in : 20 dimension : length in : 36.13 dimension : width in : 36.13', explanation='63.67524099349

## Describe the search tool to the LLM

There is a specific schema for telling OpenAI about our tools / functions. However, the cheat at search library has added some conveniences:

* We use the function name as the name to OpenAI
* We use the doc string to get a description
* The typing information gets encoded in parameters and return value

So IMPORTANTLY -- all these things are part of the prompt

### Annoying serialization / deserialization

When we get it in an OpenAI-friendly format, we also keep around some book-keeping for annoying serialization / deserialization of the arguments

With this we get some plumbing information in a 3-tuple
* The arguments to pass (as one pydantic struct)
* The tool as OpenAI sees it
* The function to call to delegate to this tool

Don't get too lost in the weeds here. In future notebooks, cheat-at-search helper code will just do this for you behind the scenes.

In [ ]:
from cheat_at_search.agent.pydantize import make_tool_adapter
search_tool = make_tool_adapter(search_furniture)
score_tool = make_tool_adapter(score)

tool_info = {search_furniture.__name__: search_tool,
             score.__name__: score_tool}
tool_info

{'search_furniture': (cheat_at_search.agent.pydantize.Search_furnitureArgs,
  {'type': 'function',
   'name': 'search_furniture',
   'description': "Search the available furniture products, get top 10 furniture.\n\n    This is just a naive BM25 / keyword search of the product title and description.\n    Don't expect sophisticated synonyms or semantic search. Just basic keyword with\n    some stemming.",
   'parameters': {'properties': {'keywords': {'title': 'Keywords',
      'type': 'string'}},
    'required': ['keywords'],
    'title': 'Search_furnitureArgs',
    'type': 'object'}},
  <function cheat_at_search.agent.pydantize.make_tool_adapter.<locals>.call_from_tool(d: dict)>),
 'score': (cheat_at_search.agent.pydantize.ScoreArgs,
  {'type': 'function',
   'name': 'score',
   'description': "Return whether the results for the query will satisfy the user\n\n    * If you call 'score' on a result, and it doesn't have a user_satisfaction label, its unknown whether the user would like it\

## Issue the requested calls to your search tool

Now we do the magic of calling the tools directly

You can ignore the `annoying_tool_marshalling` its doing some lookups and plumbings to go between the JSON arguments and the Python world we have here.

The important thing is that we recieve a tool call request, we call the requested tool (here by doing a lookup and getting `tool_fn` that just wraps the search function)

Then we go on to append those all back into th inputs, with a JSON response, the call id, and a note to OpenAI this is a "function_call_output"

In [ ]:
def annoying_tool_marshalling(item) -> dict:

    # Lookup how the agent wants to call the tool
    tool_name = item.name
    tool = tool_info[tool_name]
    ToolArgsModel = tool[0]
    tool_fn = tool[2]
    fn_args: ToolArgsModel = ToolArgsModel.model_validate_json(item.arguments)

    # The tool call function itself (ie search)
    # wrapped in something helping with serialization
    print(f"Calling {tool_name} with {fn_args}")
    py_resp, json_resp = tool_fn(fn_args)

    # 4. Provide function call results to the model
    return {
        "type": "function_call_output",
        "call_id": item.call_id,
        "output": json_resp,
    }



## Put it all in one loop

In [ ]:
import textwrap
from pydantic import BaseModel, Field

system_prompt = """
Users are searching a catalog of furniture, and you're going to help them find
results that satisfies them!

* Use the search tool (search_furniture) to retrieve search results (notice its limitations)
* Use the scoring tool (score) to see if this will satisfy them

We're a poor, small company, so our search is not good. It's just a keyword search. We need you,
tireless agent, to use trial and error with our search tool!

Use trial and error with the search tool to find items that satisfy users. Try to find 10 results.

If you only find results that will actively dissatisfy the user, just omit those results. Even 0 results!

Any calls to search_furniture that yield results that seem to satisfy users, put them in the tool_queries_satisfying_user list.

"""
# Finally, for tool_reflection, give generic, broadly applicable guidance regardless of the query. We need to take that and use it for any query.

def agentic_search(query: str, hint=None, summary=True) -> str:

    inputs = []
    inputs.append({"role": "system", "content": system_prompt})

    if hint:
        inputs.append({"role": "user", "content": "Here is a hint: " + hint})

    inputs.append({"role": "user", "content": "The users query:" + query})


    tool_calls = True
    resp = None
    while tool_calls:
        print("Calling OpenAI")
        resp = openai.responses.parse(
            model="gpt-5",
            input=inputs,
            tools=[tool[1] for tool in tool_info.values()],
            reasoning={
                "effort": "medium",
                "summary": "auto" if summary else "none"
            },
            text_format=ScoredSearchResults
        )
        print("...done")
        inputs += resp.output
        if summary:
            print("\n## Reasonings: ")
            for item in resp.output:
                if item.type == "reasoning":
                    for summary_item in item.summary:
                        print(textwrap.fill(summary_item.text, 80), "\n")
                    item.summary = []

        for item in resp.output:
            tool_calls = False
            if item.type == "function_call":
                tool_calls = True
                # *** Get the tool, and package
                # up the call to the tool (our python function)
                tool_response = annoying_tool_marshalling(item)

                # 4. Provide function call results to the model
                inputs.append(tool_response)
    return resp.output_parsed


results = agentic_search('small ladies rocker swivel recliner')

Calling OpenAI
...done

## Reasonings: 
**Searching for furniture options**  I'm getting ready to use functions to
search for furniture and I think I'll need to experiment with different keyword
variations since the search method is pretty basic. The goal is to find a top 10
list of "small ladies rocker swivel recliners." I'm considering multiple options
like "swivel recliner," "rocker recliner," and a few others to improve the
outcomes. I want to make sure the scores tell me how satisfying the results are
for the user. Let’s see how this works! 

**Using the score tool**  The score tool needs both a query string and a results
list, so I’ll pass the furniture search results into it. I want to try multiple
search queries and then score each one to see which results are satisfying for
the user. It looks like the score returns a satisfaction label for overall
results or per result. The expected output format seems to be
ScoredSearchResults, which includes user satisfaction emojis for each

In [ ]:
print("msg", textwrap.fill(results.message, 80))
hint = results.tool_reflection
print("reflection", textwrap.fill(results.tool_reflection, 80))
for result in results.search_results:
    print(f"{result.user_satisfaction} -- {result.result.title} ({result.result.doc_id})")

msg I couldn’t find an exact “small ladies rocker swivel recliner” that clearly
mentions petite sizing and all three motions together. Here are the closest
matches that swivel, rock, and recline (or are compact rockers). If you can
share ideal seat width/height or max overall width (e.g., under 28–30 inches), I
can retry with tighter filters to find a better petite fit.
reflection Our keyword search struggles with nuanced intents like “petite/ladies.” To
improve results: try multiple synonyms and constraints (e.g., “28 wide,” “25
wide,” “compact,” “petite,” “nursery glider,” “swivel rocking recliner”).
Include motion words explicitly (swivel, rocker, glider, recliner). Add size
keywords in inches to surface narrower models.
🫤 -- arlo swivel reclining rocking glider (9210)
🫤 -- ruby swivel reclining glider (11230)
🫤 -- 30.5 '' wide manual glider club recliner (31267)
🫤 -- vondrus 32.75 '' wide manual rocker standard recliner (40172)


In [ ]:
results.tool_queries_satisfying_user

[]

In [ ]:
results = agentic_search('small ladies rocker swivel recliner',
                         hint=hint)

Calling OpenAI
...done

## Reasonings: 
**Executing targeted searches**  I need to conduct searches using the provided
tools based on the user's instructions, focusing on specific keyword variations
related to furniture like swivel, rocker, glider, and recliner. I'll include
size cues in inches. My plan is to score each batch, keeping only the
satisfactory results and deduping them by doc_id. I can keep iterating until I
get some positive results. I’ll use our two tools, search_furniture and score,
to ensure user satisfaction with the results. 

**Planning keyword search strategy**  I’m working with the ScoredSearchResults
schema, which provides properties like user satisfaction ratings. The score tool
returns results based on this schema. I need to iterate various keyword
combinations to aim for around 10 satisfactory results. If I find any that lead
to dissatisfaction, I should omit them. The user has suggested avoiding the term
"ladies," so I'll focus on "petite" and "small" instead

In [ ]:
print("msg", textwrap.fill(results.message, 80))
hint = results.tool_reflection
print("reflection", textwrap.fill(results.tool_reflection, 80))
for result in results.search_results:
    print(f"{result.user_satisfaction} -- {result.result.title} ({result.result.doc_id})")

msg I ran multiple targeted searches with function terms (swivel, rocker, glider,
recliner) plus size cues (petite/small and widths like 25–33 inches). While I
found several compact options, none scored as a perfect match (😁) for “small
ladies rocker swivel recliner.” The best near-matches (kept here) are
petite/narrow rocker-recliners (no swivel) and compact swivel glider recliners
(glide/rock + swivel). If swivel is a must in addition to rocking, would a
swivel glider recliner work? Also, what maximum overall width are you targeting
(e.g., ≤28–30")? With that, I can refine and try again for closer hits.
reflection Issue multiple narrow keyword queries with exact function words and quoted inch
widths (e.g., 28" wide swivel rocker recliner). Avoid terms like “ladies” that
introduce decor noise. Always score each batch, keep only non-😥, and dedupe by
doc_id. Iterate with size synonyms (petite, small, compact, narrow) and function
variants (glider vs rocker) to surface acceptable alterna

In [ ]:
from cheat_at_search.wands_data import judgments

judgments[judgments['query'] == 'small ladies rocker swivel recliner']

,query_id,query,query_class,doc_id,id,product_id,label,grade
175412,400,small ladies rocker swivel recliner,Recliners,2266,41010,2266,Partial,1.0
175413,400,small ladies rocker swivel recliner,Recliners,2517,157487,2517,Partial,1.0
175414,400,small ladies rocker swivel recliner,Recliners,2636,157479,2636,Partial,1.0
175415,400,small ladies rocker swivel recliner,Recliners,9210,157489,9210,Partial,1.0
175416,400,small ladies rocker swivel recliner,Recliners,9420,41013,9420,Partial,1.0
175417,400,small ladies rocker swivel recliner,Recliners,9492,157476,9492,Partial,1.0
175418,400,small ladies rocker swivel recliner,Recliners,11115,157480,11115,Partial,1.0
175419,400,small ladies rocker swivel recliner,Recliners,11230,157483,11230,Partial,1.0
175420,400,small ladies rocker swivel recliner,Recliners,14623,157493,14623,Partial,1.0
175421,400,small ladies rocker swivel recliner,Recliners,17222,41011,17222,Partial,1.0
